# Mohammad Amin Kiani - 4043644008
# NLP - HW3 : Recurrent To Attention
# ui.ac.ir 404-405

1

In [1]:
# # نصب کتابخانه‌های مورد نیاز برای پردازش مدل‌های زبانی، کوانتیزه‌سازی و تنظیم دقیق
!pip install -q -U transformers accelerate bitsandbytes peft datasets trl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.2 MB/s eta 0:00:00


2

In [1]:
# نصب نسخه‌های پایدار و هماهنگ کتابخانه‌ها
!pip install -q -U "transformers>=4.40.0" "accelerate>=0.28.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" datasets sklearn

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


3

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" "transformers>=4.40.0" accelerate peft datasets scikit-learn tqdm

#### 1:

In [5]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# نام مدل سبک که انتخاب کردیم
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# تنظیمات پرامپت‌ها برای سه وظیفه
tasks = {
    "Translation": "متن زیر را به انگلیسی ترجمه کن:\nدیروز به کتابخانه رفتم تا یک کتاب جدید برای مطالعه پیدا کنم.",
    "Summarization": "متن زیر را در یک جمله خلاصه کن:\nهوش مصنوعی در سال‌های اخیر پیشرفت‌های چشمگیری داشته است. الگوریتم‌های یادگیری عمیق توانسته‌اند در پردازش تصویر و زبان طبیعی به سطح انسان نزدیک شوند و این موضوع باعث تحول در صنایع مختلف از جمله پزشکی و خودروسازی شده است.",
    "Sentiment": "احساس متن زیر را مشخص کن (فقط یک کلمه بنویس: مثبت، منفی یا خنثی):\nمحصولی که به دستم رسید کیفیت فوق‌العاده‌ای داشت و بسته‌بندی آن بسیار زیبا بود."
}

# تابعی برای تعریف تنظیمات کوانتیزه‌سازی بر اساس نام حالت
def get_quantization_config(precision):
    if precision == "FP16":
        return None # کوانتیزه نمی‌شود، با فرمت ۱۶ بیت می‌آید
    elif precision == "INT8":
        return BitsAndBytesConfig(load_in_8bit=True)
    elif precision == "NF4":
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    elif precision == "INT4":
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="fp4", bnb_4bit_compute_dtype=torch.float16)

# تابعی برای ارزیابی مدل
def evaluate_model(precision):
    print(f"\n{'='*40}\nEvaluating precision: {precision}\n{'='*40}")

    # ریست کردن حافظه GPU برای اندازه‌گیری دقیق‌تر
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    quant_config = get_quantization_config(precision)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # بارگذاری مدل با کانفیگ مربوطه
    if precision == "FP16":
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, device_map="auto")

    # محاسبه حداکثر حافظه مصرفی (VRAM)
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3) # تبدیل بایت به گیگابایت
    print(f"Peak VRAM Usage: {peak_vram:.2f} GB")

    # اجرای وظایف
    for task_name, prompt in tasks.items():
        # تبدیل پرامپت به فرمت چت استاندارد مدل
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)

        start_time = time.time()

        # تولید متن با تنظیمات خواسته‌شده توسط تمرین
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0,
            do_sample=False
        )

        end_time = time.time()
        latency = end_time - start_time

        # محاسبه تعداد توکن‌های تولید شده (فقط توکن‌های جدید، بدون پرامپت)
        generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        num_tokens = len(generated_tokens)
        tokens_per_sec = num_tokens / latency

        # استخراج خروجی متنی
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        print(f"\n--- Task: {task_name} ---")
        print(f"Latency: {latency:.2f} sec | Tokens/sec: {tokens_per_sec:.2f}")
        print(f"Output: {response.strip()}")

    # پاک کردن مدل از حافظه برای دور بعدی
    del model
    del tokenizer
    torch.cuda.empty_cache()

# اجرای حلقه روی تمام حالت‌ها
precisions = ["FP16", "INT8", "NF4", "INT4"]
for p in precisions:
    evaluate_model(p)


Evaluating precision: FP16


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Peak VRAM Usage: 3.30 GB

--- Task: Translation ---
Latency: 0.68 sec | Tokens/sec: 22.16
Output: Doris wrote to the library to find a new book for research.

--- Task: Summarization ---
Latency: 5.70 sec | Tokens/sec: 22.46
Output: در سال‌های اخیر، هوش مصنوعی پیشرفت‌های چشمگیری داشت و الگوریتم‌های یادگیری عمیق همراه بودند. این موضوع باعث تحول در صنایع مختلف، از پزشکی به خودروسازی، به منظور کاهش آثار مشکلات و توسعه فناوری‌ها، به روزرسانی و تحسین عملکرد و کاربردهای

--- Task: Sentiment ---
Latency: 0.17 sec | Tokens/sec: 23.84
Output: منفی

Evaluating precision: INT8


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Peak VRAM Usage: 2.98 GB


...


--- Task: Sentiment ---
Latency: 1.89 sec | Tokens/sec: 2.12
Output: منفی

Evaluating precision: NF4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Peak VRAM Usage: 2.83 GB

--- Task: Translation ---
Latency: 2.41 sec | Tokens/sec: 10.35
Output: Here is the translation of that sentence into English:

I sent Droid to search for a new book to study.
)

--- Task: Summarization ---
Latency: 11.89 sec | Tokens/sec: 10.76
Output: در اینجا یک جمله برای یافتن یک زمانی زیادی برای یافتن یک یادگیری گذاری یا گروه گذاری یا یک گروه گفاهای یا گروه گفاهای یا گروه گفاهای یا گروه گفاهای یا گروه گفاهای یا گروه

--- Task: Sentiment ---
Latency: 0.40 sec | Tokens/sec: 10.00
Output: منفی

Evaluating precision: INT4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Peak VRAM Usage: 3.02 GB

--- Task: Translation ---
Latency: 6.38 sec | Tokens/sec: 10.50
Output: Here's the English translation of the given text:

I am looking for a book that I can read to learn about the world.

This is an English version of the original text in simple English. The original text was in Persian and it translates to "I am searching for a book that I can read to learn about the world."

--- Task: Summarization ---
Latency: 11.43 sec | Tokens/sec: 11.20
Output: برخه، معمولاً: "هوش مصنوعی در سال‌های اخیر پیشرفت‌های چشمگیری داشته است. الگوریتم‌های یادگیری عمیق توانسته‌اند در پردازش تصویر و زبان طبیعی به سطح انسان نزدیک شوند و این موضوع باعث تحول در صنایع مختلف از جمله پزشکی و خودرو

--- Task: Sentiment ---
Latency: 0.37 sec | Tokens/sec: 10.68
Output: منفی


#### 2:

In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# ۱. بارگذاری مجموعه داده
print("Loading Dataset...")
dataset = load_dataset("hezarai/sentiment-dksf")

train_data = dataset['train']
test_data = dataset['test']
print(f"Train size: {len(train_data)} | Test size: {len(test_data)}")

# ۲. آماده‌سازی توکنایزر
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

label_map = {0: "منفی", 1: "خنثی", 2: "مثبت"}

# تابع پردازش و توکنایز کردن داده‌ها
def preprocess_function(example):
    prompt = f"متن: {example['text']}\nاحساس:"
    label_text = label_map.get(example['label'], "")

    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": label_text}
    ]

    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    return tokenizer(formatted_text, truncation=True, max_length=512)

print("Tokenizing training data...")
tokenized_train_data = train_data.map(preprocess_function, remove_columns=train_data.column_names)

# ۳. بارگذاری مدل پایه به صورت NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

# ۴. تنظیمات LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

print("Trainable Parameters:")
model.print_trainable_parameters()

# ۵. تنظیمات آموزش (Training Arguments)
training_args = TrainingArguments(
    output_dir="./qlora_sentiment",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    fp16=True,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ۶. تنظیم Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_train_data,
    args=training_args,
    data_collator=data_collator
)

# شروع آموزش
print("Starting Training...")
trainer.train()

# ۷. ذخیره فقط آداپتور (Adapter)
trainer.model.save_pretrained("adapter_qlora")
print("Adapter saved successfully in 'adapter_qlora' folder.")



Loading Dataset...


README.md:   0%|          | 0.00/705 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.44MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  266kB            

...

Trainable Parameters:
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
Starting Training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,2.112621
100,1.931543
150,1.795598
200,1.820390
250,1.806850
300,1.752886
350,1.819874
400,1.668759
450,1.683649
500,1.697593


Step,Training Loss
50,2.112621
100,1.931543
150,1.795598
200,1.820390
250,1.806850
300,1.752886
350,1.819874
400,1.668759
450,1.683649
500,1.697593


Adapter saved successfully in 'adapter_qlora' folder.


In [2]:
# ==========================================
# ارزیابی روی داده‌های تست و محاسبه معیارهای صحت
# ==========================================

def evaluate_test_set(trained_model, tokenizer, test_dataset, num_samples=500):
    test_subset = test_dataset.select(range(min(num_samples, len(test_dataset))))

    true_labels = []
    pred_labels = []
    errors = []

    for row in tqdm(test_subset):
        prompt = f"متن: {row['text']}\nاحساس:"
        messages = [{"role": "user", "content": prompt}]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(input_text, return_tensors="pt").to(trained_model.device)

        output = trained_model.generate(**inputs, max_new_tokens=10, temperature=0.0, do_sample=False)
        gen_text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

        pred = gen_text.replace(":", "").strip()
        true_label = label_map.get(row['label'], "")

        true_labels.append(true_label)
        pred_labels.append(pred)

        if pred != true_label and len(errors) < 5:
            errors.append({"text": row['text'], "true": true_label, "pred": pred})

    acc = accuracy_score(true_labels, pred_labels)
    labels_list = ["مثبت", "منفی", "خنثی"]
    f_macro = f1_score(true_labels, pred_labels, labels=labels_list, average='macro', zero_division=0)

    return acc, f_macro, errors

print("\nEvaluating trained model...")
acc, f_macro, error_samples = evaluate_test_set(trainer.model, tokenizer, test_data)
print(f"Accuracy: {acc:.4f} | F-Macro: {f_macro:.4f}")

print("\n--- Error Samples for Report ---")
for e in error_samples:
    print(f"Text: {e['text']}\nTrue: {e['true']} | Predicted: {e['pred']}\n-")


Evaluating trained model...


  0%|          | 0/500 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the 

Accuracy: 0.0000 | F-Macro: 0.0000

--- Error Samples for Report ---
Text: عطره و بیشتر به درد لباس میخوره، چون اصلا پخش بو نداره و ظرف ۴-۵ روز تبخیر میشه، بوی خوبی داره به شرطی که جلوی بینی بگیرید
True: منفی | Predicted: منsystemsystemsystemsystemsystemsystemsystemsystemsystem
-
Text: اصلا بدرد نمیخوره.بعد از دو سه روز کنده میشه.زود کثیف میشه و کثیفی رو بخودش میگیره.
True: منفی | Predicted: منsystemsystemsystemsystemsystemsystemsystemsystemsystem
-
Text: غذا بسیار با کیفیت بود اما متاسفانه دوتا سفارش کباب لقمه و کباب بختیاری رو داخل یک پک غذا قرار داده بودن و این صرفه جویی جالب نبود. اما مخلفات و طعم غذا عالی بود
True: خنثی | Predicted: خsystemsystemsystemsystemsystemsystemsystemsystemsystem
-
Text: غذا خیلی خوب بود، خیلی سریع رسید و قیمتشم خیلی مناسب بود نسبت به باقی رستورانهای این اطراف.
True: خنثی | Predicted: خsystemsystemsystemsystemsystemsystemsystemsystemsystem
-
Text: قیمت رو بالا ببرید کیفیت رو کم نکنید لطفا
True: خنثی | Predicted: منsystemsystemsystemsystemsystemsystemsystem

In [8]:
!pip install -q -U torchao peft transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 37.4 MB/s eta 0:00:00


In [9]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading Dataset and Tokenizer...")
dataset = load_dataset("hezarai/sentiment-dksf")
test_data = dataset['test']

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

label_map = {0: "منفی", 1: "خنثی", 2: "مثبت"}

print("Loading Base Model and Trained QLoRA Adapter...")
# بارگذاری مدل پایه (استفاده از dtype به جای torch_dtype جهت رفع هشدار)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# بارگذاری مستقیم آداپتور ذخیره‌شده QLoRA
model = PeftModel.from_pretrained(base_model, "adapter_qlora")
model.eval()

# تابع هوشمند برای استخراج و نگاشت خروجی مدل به کلاس صحیح
def clean_and_extract_label(gen_text):
    gen_text = gen_text.strip()
    if "مثبت" in gen_text or gen_text.startswith("مث"):
        return "مثبت"
    elif "منفی" in gen_text or gen_text.startswith("من"):
        return "منفی"
    elif "خنثی" in gen_text or gen_text.startswith("خ"):
        return "خنثی"
    return gen_text # در صورت عدم تطابق، همان متن خام برمی‌گردد

def evaluate_test_set_fixed(model, tokenizer, test_dataset, num_samples=500):
    test_subset = test_dataset.select(range(min(num_samples, len(test_dataset))))

    true_labels = []
    pred_labels = []
    errors = []

    for row in tqdm(test_subset, desc="Evaluating QLoRA"):
        prompt = f"متن: {row['text']}\nاحساس:"
        messages = [{"role": "user", "content": prompt}]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=5,             # کاهش توکن‌ها برای جلوگیری از زیاده‌گویی مدل
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        gen_text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = clean_and_extract_label(gen_text)
        true_label = label_map.get(row['label'], "")

        true_labels.append(true_label)
        pred_labels.append(pred)

        if pred != true_label and len(errors) < 5:
            errors.append({"text": row['text'], "true": true_label, "pred": pred, "raw": gen_text})

    acc = accuracy_score(true_labels, pred_labels)
    labels_list = ["مثبت", "منفی", "خنثی"]
    f_macro = f1_score(true_labels, pred_labels, labels=labels_list, average='macro', zero_division=0)

    return acc, f_macro, errors

print("\nEvaluating QLoRA trained model with fixed logic...")
acc, f_macro, error_samples = evaluate_test_set_fixed(model, tokenizer, test_data)

print(f"\n==========================================")
print(f"QLoRA Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"QLoRA F-Macro  : {f_macro:.4f}")
print(f"==========================================")

print("\n--- Error Samples for Report ---")
for e in error_samples:
    print(f"Text: {e['text']}\nTrue: {e['true']} | Predicted: {e['pred']} (Raw Output: '{e['raw']}')\n-")

Loading Dataset and Tokenizer...
Loading Base Model and Trained QLoRA Adapter...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Evaluating QLoRA trained model with fixed logic...


Evaluating QLoRA: 100%|██████████| 500/500 [03:25<00:00,  2.43it/s]


QLoRA Accuracy : 0.8340 (83.40%)
QLoRA F-Macro  : 0.7044

--- Error Samples for Report ---
Text: قیمت رو بالا ببرید کیفیت رو کم نکنید لطفا
True: خنثی | Predicted: منفی (Raw Output: 'منفی و خ')
-
Text: نونش تازه نبود وکبابش کیفیتش امده بود پایین
True: خنثی | Predicted: منفی (Raw Output: 'منفی هست')
-
Text: متاسفانه محصولی که فرستادن قرق داشت توصیه میکنم COBجدا بگیرید دستی جایگزین کنید
True: خنثی | Predicted: منفی (Raw Output: 'منفی و خ')
-
Text: خیلی بوش قدیمیه و شبیه ادو کلن های سی سال پیشه قیمتشم موقع خرید شگفت انگیز بالاتر بود و تخفیف زده بود ! در کل تو‌شگفت انگیز بد نیست
True: مثبت | Predicted: منفی (Raw Output: 'منفی و ر')
-
Text: قطر و کتف خوبی داره اندازه مناسبداخلش هم مواد هست که خب طبیعتا با مرور زمان میخوابه ولی می ارزه
True: خنثی | Predicted: منفی (Raw Output: 'منفی که')
-


#### 3:

In [ ]:
import torch
import time
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import PrefixTuningConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# ۱. بارگذاری دیتاست و توکنایزر
print("Loading Dataset and Tokenizer...")
dataset = load_dataset("hezarai/sentiment-dksf")
train_data = dataset['train']
test_data = dataset['test']

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
label_map = {0: "منفی", 1: "خنثی", 2: "مثبت"}

# تابع پردازش داده‌ها
def preprocess_function(example):
    prompt = f"متن: {example['text']}\nاحساس:"
    label_text = label_map.get(example['label'], "")
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": label_text}
    ]
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    return tokenizer(formatted_text, truncation=True, max_length=512)

print("Tokenizing data...")
tokenized_train_data = train_data.map(preprocess_function, remove_columns=train_data.column_names)

# ۲. بارگذاری مدل پایه و تنظیمات Prefix Tuning
print("\nLoading Model for Prefix Tuning...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_pt = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

# غیرفعال کردن Gradient Checkpointing جهت سازگاری با Prefix Tuning
model_pt = prepare_model_for_kbit_training(model_pt, use_gradient_checkpointing=False)

prefix_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=20
)

model_pt = get_peft_model(model_pt, prefix_config)

print("\nPrefix Tuning Trainable Parameters:")
model_pt.print_trainable_parameters()

# ۳. تنظیمات آموزش (Training Arguments)
training_args_pt = TrainingArguments(
    output_dir="./prefix_sentiment",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer_pt = Trainer(
    model=model_pt,
    train_dataset=tokenized_train_data,
    args=training_args_pt,
    data_collator=data_collator
)

# ۴. شروع آموزش و محاسبه زمان
print("\nStarting Prefix Tuning Training...")
start_pt_train = time.time()

trainer_pt.train()

end_pt_train = time.time()
pt_training_time = end_pt_train - start_pt_train
print(f"\n==========================================")
print(f"Prefix Tuning Training Time: {pt_training_time:.2f} seconds ({pt_training_time/60:.2f} minutes)")
print(f"==========================================")

# ۵. ذخیره آداپتور Prefix Tuning
trainer_pt.model.save_pretrained("adapter_prefix")
print("Adapter saved successfully in 'adapter_prefix' folder.")


# ==========================================
# ۶. ارزیابی مدل Prefix Tuning روی داده‌های تست
# ==========================================

def clean_and_extract_label(gen_text):
    gen_text = gen_text.strip()
    if "مثبت" in gen_text or gen_text.startswith("مث"):
        return "مثبت"
    elif "منفی" in gen_text or gen_text.startswith("من"):
        return "منفی"
    elif "خنثی" in gen_text or gen_text.startswith("خ"):
        return "خنثی"
    return gen_text

def evaluate_prefix_model(model, tokenizer, test_dataset, num_samples=500):
    test_subset = test_dataset.select(range(min(num_samples, len(test_dataset))))

    true_labels = []
    pred_labels = []
    errors = []

    model.eval()
    for row in tqdm(test_subset):
        prompt = f"متن: {row['text']}\nاحساس:"
        messages = [{"role": "user", "content": prompt}]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        gen_text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = clean_and_extract_label(gen_text)
        true_label = label_map.get(row['label'], "")

        true_labels.append(true_label)
        pred_labels.append(pred)

        if pred != true_label and len(errors) < 5:
            errors.append({"text": row['text'], "true": true_label, "pred": pred, "raw": gen_text})

    acc = accuracy_score(true_labels, pred_labels)
    labels_list = ["مثبت", "منفی", "خنثی"]
    f_macro = f1_score(true_labels, pred_labels, labels=labels_list, average='macro', zero_division=0)

    return acc, f_macro, errors

print("\nEvaluating Prefix Tuning Trained Model...")
acc_pt, f_macro_pt, error_samples_pt = evaluate_prefix_model(trainer_pt.model, tokenizer, test_data)

print(f"\n==========================================")
print(f"Prefix Tuning Accuracy : {acc_pt:.4f} ({acc_pt*100:.2f}%)")
print(f"Prefix Tuning F-Macro  : {f_macro_pt:.4f}")
print(f"==========================================")

print("\n--- Error Samples (Prefix Tuning) ---")
for e in error_samples_pt:
    print(f"Text: {e['text']}\nTrue: {e['true']} | Predicted: {e['pred']} (Raw: '{e['raw']}')\n-")

Loading Dataset and Tokenizer...


README.md:   0%|          | 0.00/705 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.44MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  266kB            

...


Prefix Tuning Trainable Parameters:
trainable params: 122,880 || all params: 494,155,648 || trainable%: 0.0249

Starting Prefix Tuning Training...


Step,Training Loss
50,12.735897
100,10.815051
150,9.058045
200,7.699527
250,6.678441
300,5.906814
350,5.501271
400,5.020912
450,4.753682
500,4.626589



Prefix Tuning Training Time: 2370.23 seconds (39.50 minutes)
Adapter saved successfully in 'adapter_prefix' folder.

Evaluating Prefix Tuning Trained Model...


100%|██████████| 500/500 [02:22<00:00,  3.50it/s]


Prefix Tuning Accuracy : 0.3080 (30.80%)
Prefix Tuning F-Macro  : 0.2609

--- Error Samples (Prefix Tuning) ---
Text: عطره و بیشتر به درد لباس میخوره، چون اصلا پخش بو نداره و ظرف ۴-۵ روز تبخیر میشه، بوی خوبی داره به شرطی که جلوی بینی بگیرید
True: منفی | Predicted: فیسای ک (Raw: 'فیسای ک')
-
Text: اصلا بدرد نمیخوره.بعد از دو سه روز کنده میشه.زود کثیف میشه و کثیفی رو بخودش میگیره.
True: منفی | Predicted: احساس گون (Raw: 'احساس گون')
-
Text: قیمت رو بالا ببرید کیفیت رو کم نکنید لطفا
True: خنثی | Predicted: احساس: 2 (Raw: 'احساس: 2')
-
Text: حدود دو سال پیش از ترکیه خریدم. عالیه. قسمتی که کودک درونش قرار می گیرد جدا شده و در نوزادی من به عنوان کریر از آن استفاده کردم. اکنون که فرزندم بیست ماهشه ، به شکل نشسته در میاورم و هر وقت خوابید به شکل خوابیده. طوری ساخته شده که وقتی باد بوزد، کودک از باد حفظ است. با کاوری که دارد در باران هم قطره ای آب نفوذ نمی کند. بدنه بسیار مستحکم است. چرخها محکم و به راحتی دور زده میشوند بدون اینکه کالسکه را بلند کنم. صندلی از نظر ارگونامی استاندارد است و به فر

In [4]:
import torch
import time
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import PrefixTuningConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# ۱. بارگذاری دیتاست و توکنایزر
print("Loading Dataset and Tokenizer...")
dataset = load_dataset("hezarai/sentiment-dksf")
train_data = dataset['train']
test_data = dataset['test']

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
label_map = {0: "منفی", 1: "خنثی", 2: "مثبت"}

# ۲. تابع پردازش داده‌ها با ماسک‌گذاری هوشمند پرامپت (Masking)
def preprocess_function_masked(example):
    prompt = f"متن: {example['text']}\nاحساس:"
    label_text = label_map.get(example['label'], "")

    # ساخت پرامپت ورودی کاربر
    user_msg = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(user_msg, tokenize=False, add_generation_prompt=True)

    # متن پاسخ هدف
    target_text = f"{label_text}{tokenizer.eos_token}"

    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    target_ids = tokenizer.encode(target_text, add_special_tokens=False)

    # ترکیب ورودی‌ها
    input_ids = prompt_ids + target_ids
    # قرار دادن 100- برای پرامپت جهت عدم محاسبه خطا روی متن ورودی
    labels = [-100] * len(prompt_ids) + target_ids

    # برش داده‌های طولانی
    if len(input_ids) > 512:
        input_ids = input_ids[-512:]
        labels = labels[-512:]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels
    }

print("Tokenizing and Masking data...")
tokenized_train_data = train_data.map(
    preprocess_function_masked,
    remove_columns=train_data.column_names,
    desc="Processing Train Data"
)

# ۳. بارگذاری مدل پایه و تنظیمات Prefix Tuning
print("\nLoading Model for Prefix Tuning...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_pt = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
model_pt = prepare_model_for_kbit_training(model_pt, use_gradient_checkpointing=False)

# افزایش تعداد توکن‌های مجازی به ۳۰ برای یادگیری بهتر
prefix_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=30
)

model_pt = get_peft_model(model_pt, prefix_config)

print("\nPrefix Tuning Trainable Parameters:")
model_pt.print_trainable_parameters()

# ۴. تنظیمات آموزش بهینه‌شده
training_args_pt = TrainingArguments(
    output_dir="./prefix_sentiment_optimized",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-3,             # افزایش نرخ یادگیری مخصوص Prefix Tuning
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True,
    report_to="none"
)

# استفاده از DataCollator مناسب برای پدینگ لیبل‌ها با 100-
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True
)

trainer_pt = Trainer(
    model=model_pt,
    train_dataset=tokenized_train_data,
    args=training_args_pt,
    data_collator=data_collator
)

# ۵. شروع آموزش و محاسبه زمان
print("\nStarting Optimized Prefix Tuning Training...")
start_pt_train = time.time()

trainer_pt.train()

end_pt_train = time.time()
pt_training_time = end_pt_train - start_pt_train
print(f"\n==========================================")
print(f"Prefix Tuning Training Time: {pt_training_time:.2f} seconds ({pt_training_time/60:.2f} minutes)")
print(f"==========================================")

# ۶. ذخیره آداپتور اصلاح‌شده
trainer_pt.model.save_pretrained("adapter_prefix_optimized")
print("Optimized adapter saved successfully in 'adapter_prefix_optimized' folder.")


# ==========================================
# ۷. ارزیابی مدل بهینه‌شده Prefix Tuning
# ==========================================

def clean_and_extract_label(gen_text):
    gen_text = gen_text.strip()
    if "مثبت" in gen_text or gen_text.startswith("مث"):
        return "مثبت"
    elif "منفی" in gen_text or gen_text.startswith("من"):
        return "منفی"
    elif "خنثی" in gen_text or gen_text.startswith("خ"):
        return "خنثی"
    return gen_text

def evaluate_prefix_model(model, tokenizer, test_dataset, num_samples=500):
    test_subset = test_dataset.select(range(min(num_samples, len(test_dataset))))

    true_labels = []
    pred_labels = []
    errors = []

    model.eval()
    for row in tqdm(test_subset, desc="Evaluating"):
        prompt = f"متن: {row['text']}\nاحساس:"
        messages = [{"role": "user", "content": prompt}]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        gen_text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = clean_and_extract_label(gen_text)
        true_label = label_map.get(row['label'], "")

        true_labels.append(true_label)
        pred_labels.append(pred)

        if pred != true_label and len(errors) < 5:
            errors.append({"text": row['text'], "true": true_label, "pred": pred, "raw": gen_text})

    acc = accuracy_score(true_labels, pred_labels)
    labels_list = ["مثبت", "منفی", "خنثی"]
    f_macro = f1_score(true_labels, pred_labels, labels=labels_list, average='macro', zero_division=0)

    return acc, f_macro, errors

print("\nEvaluating Optimized Prefix Tuning Model...")
acc_pt, f_macro_pt, error_samples_pt = evaluate_prefix_model(trainer_pt.model, tokenizer, test_data)

print(f"\n==========================================")
print(f"Optimized Prefix Tuning Accuracy : {acc_pt:.4f} ({acc_pt*100:.2f}%)")
print(f"Optimized Prefix Tuning F-Macro  : {f_macro_pt:.4f}")
print(f"==========================================")

print("\n--- Error Samples (Optimized Prefix Tuning) ---")
for e in error_samples_pt:
    print(f"Text: {e['text']}\nTrue: {e['true']} | Predicted: {e['pred']} (Raw: '{e['raw']}')\n-")

Loading Dataset and Tokenizer...
Tokenizing and Masking data...


Processing Train Data:   0%|          | 0/28602 [00:00<?, ? examples/s]


Loading Model for Prefix Tuning...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Prefix Tuning Trainable Parameters:
trainable params: 184,320 || all params: 494,217,088 || trainable%: 0.0373

Starting Optimized Prefix Tuning Training...


Step,Training Loss
50,11.634110
100,4.888058
150,1.331167
200,0.510786
250,0.379337
300,0.337982
350,0.336870
400,0.320464
450,0.327852
500,0.319093



Prefix Tuning Training Time: 1735.32 seconds (28.92 minutes)
Optimized adapter saved successfully in 'adapter_prefix_optimized' folder.

Evaluating Optimized Prefix Tuning Model...


Evaluating: 100%|██████████| 500/500 [02:25<00:00,  3.42it/s]


Optimized Prefix Tuning Accuracy : 0.6720 (67.20%)
Optimized Prefix Tuning F-Macro  : 0.4820

--- Error Samples (Optimized Prefix Tuning) ---
Text: قیمت رو بالا ببرید کیفیت رو کم نکنید لطفا
True: خنثی | Predicted: منفی (Raw: 'منفی')
-
Text: حدود دو سال پیش از ترکیه خریدم. عالیه. قسمتی که کودک درونش قرار می گیرد جدا شده و در نوزادی من به عنوان کریر از آن استفاده کردم. اکنون که فرزندم بیست ماهشه ، به شکل نشسته در میاورم و هر وقت خوابید به شکل خوابیده. طوری ساخته شده که وقتی باد بوزد، کودک از باد حفظ است. با کاوری که دارد در باران هم قطره ای آب نفوذ نمی کند. بدنه بسیار مستحکم است. چرخها محکم و به راحتی دور زده میشوند بدون اینکه کالسکه را بلند کنم. صندلی از نظر ارگونامی استاندارد است و به فرم کمر و بدن کودک آسیب نمی زند. بسیار استاندارد و عالی است.
True: خنثی | Predicted: منفی (Raw: 'منفی')
-
Text: خیلی خیلی سرد بود و سیب زمینی با قارچ کیفیت همیشگی رو نداشت.
True: منفی | Predicted: خنثی (Raw: 'خنثی')
-
Text: مقدار حجم ساندویج و کیفیتش به نسبت اسم هایدا خیلی خیلی پایین بود البته سالهاست که